# Phase 3 — Cross-linguistic topology via persistence-diagram distances

Tests whether mBERT attention-graph persistence diagrams differ **metrically**
across English, Russian, and Spanish color vocabulary.  Complements
`phase3_comparison.ipynb` (vectorized-feature comparison) by asking:
*do the diagram shapes differ in a metric sense?*

**Hypothesis (from CLAUDE.md):** distributional patterns in language encode
culturally-specific attentional structures that show up as measurably different
*topology* in attention graphs — not just different distances, but different
shapes.

**Scope:** COLOR domain only — COSI 115a May 2026 analysis.  
Emotion and kinship are out of scope; see `bd show ph-project-mwk`.

**Primary metric:** Wasserstein-2 (W_2), H_1 homology dimension.  
Bottleneck + H_0 robustness reruns are handled in subtask blr.6.

**Inputs:**
- `data/phase3/diagram_distances/wasserstein.npz` — cached W_2 distance
  tensor, shape `(12, 12, N, N, 2)`, produced by `scripts/compute_diagram_distances.py`
  (subtask blr.2).

**Outputs:**
- `results/phase3_diagram_distances/per_domain_perm.csv`
- `results/figures/phase3_diagram_distances_per_domain_effect_heatmap.{pdf,png}`

## Setup

In [ ]:
# Ensure the repo root is on sys.path (mirrors phase3_comparison.ipynb pattern)
import sys
import pathlib

_NOTEBOOK_DIR = pathlib.Path('__file__').resolve().parent if '__file__' in dir() else pathlib.Path('.').resolve()
_REPO_ROOT = _NOTEBOOK_DIR.parent if _NOTEBOOK_DIR.name == 'notebooks' else _NOTEBOOK_DIR

if str(_REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(_REPO_ROOT))

print(f'Repo root: {_REPO_ROOT}')

In [ ]:
import warnings

import matplotlib
matplotlib.use('Agg')  # non-interactive backend for headless execution
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import pandas as pd

warnings.filterwarnings('ignore', category=DeprecationWarning)
warnings.filterwarnings('ignore', category=FutureWarning)

from replication.diagram_distances import (
    load_distance_tensor,
    per_domain_test_statistic,
    permutation_test_per_domain,
    permutation_test_per_head,
    _bh_correction,
)

# ── Constants ──────────────────────────────────────────────────────────────
REPO_ROOT   = _REPO_ROOT
RESULTS_DIR = REPO_ROOT / 'results'
FIGURES_DIR = RESULTS_DIR / 'figures'
DD_DIR      = RESULTS_DIR / 'phase3_diagram_distances'
CACHE_DIR   = REPO_ROOT / 'data' / 'phase3' / 'diagram_distances'

# ── Scope ──────────────────────────────────────────────────────────────────
# COLOR domain only — COSI 115a May 2026 rescoping.
LANGUAGES     = ['en', 'es', 'ru']
LANG_LABELS   = {'en': 'English', 'es': 'Spanish', 'ru': 'Russian'}

# Permutation test parameters
K_PERM = 10000
SEED   = 42
FDR_Q  = 0.05

N_LAYERS = 12
N_HEADS  = 12

# ── Ensure output directories exist ────────────────────────────────────────
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
DD_DIR.mkdir(parents=True, exist_ok=True)

print('Setup complete.')
print(f'Cache dir  : {CACHE_DIR}')
print(f'Figures dir: {FIGURES_DIR}')
print(f'Output dir : {DD_DIR}')

## Load cached distance tensors

Loads the Wasserstein-2 distance tensor produced by `scripts/compute_diagram_distances.py`
(subtask blr.2).  Shape: `(12, 12, N, N, 2)` — layers × heads × samples × samples × hom_dims.
We slice out the H_1 sub-tensor for the primary analysis.

In [ ]:
# NOTE: This cell requires data/phase3/diagram_distances/wasserstein.npz
# produced by the overnight blr.2 compute job.  It is intentionally
# left un-executed in the shipped notebook.

tensor_w2, meta, metric, hom_dims = load_distance_tensor(CACHE_DIR / 'wasserstein.npz')

print(f'Metric           : {metric}')
print(f'Homology dims    : {hom_dims}')
print(f'Tensor shape     : {tensor_w2.shape}  (layers × heads × N × N × dims)')
print(f'N samples        : {tensor_w2.shape[2]}')
print(f'Language counts  :')
print(meta['lang'].value_counts().to_string())

# Slice H_1 sub-tensor: shape (12, 12, N, N)
h1_idx = list(hom_dims).index(1)
tensor_w2_h1 = tensor_w2[..., h1_idx]
print(f'\nH_1 sub-tensor shape: {tensor_w2_h1.shape}')

## Per-domain permutation test

For the head-averaged distance matrix (mean across all 144 layer/head cells)
and for each individual `(layer, head)`, we ask:

> Are between-language persistence-diagram distances systematically larger
> than within-language distances?

Test statistic: `mean(between-lang pairs) − mean(within-lang pairs)`.

Two-tailed p-value with finite-K correction:
`(|null − mean(null)| ≥ |observed − mean(null)| + 1) / (K + 1)`

Effect size: z-score under the null distribution.

BH/FDR correction at q=0.05 across all 144 `(layer, head)` tests.

In [ ]:
# ── Head-averaged per-domain test ──────────────────────────────────────────
# Average the H_1 distance matrix across all (layer, head) cells, then run
# a single permutation test on the aggregate.
mean_dist_matrix = tensor_w2_h1.mean(axis=(0, 1))  # shape (N, N)

agg_result = permutation_test_per_domain(
    mean_dist_matrix, meta, K=K_PERM, seed=SEED
)

print('=== Head-averaged per-domain permutation test ===')
print(f'  Observed statistic : {agg_result["observed"]:.6f}')
print(f'  p-value            : {agg_result["p_value"]:.4f}  (two-tailed, K={K_PERM})')
print(f'  Effect size (z)    : {agg_result["effect_size"]:.4f}')
print(f'  Null mean          : {agg_result["null"].mean():.6f}')
print(f'  Null std           : {agg_result["null"].std():.6f}')

In [ ]:
# ── Per-(layer, head) permutation test with BH correction ──────────────────
print(f'Running per-(layer, head) permutation test  (K={K_PERM}, seed={SEED}) ...')
per_head_df = permutation_test_per_head(tensor_w2_h1, meta, K=K_PERM, seed=SEED)

n_sig = per_head_df['passes_bh'].sum()
print(f'Done.  {n_sig} / {len(per_head_df)} (layer, head) cells pass BH at q={FDR_Q}.')
print(per_head_df.sort_values('effect_size', ascending=False).head(10).to_string(index=False))

## Per-(layer, head) effect-size heatmap

12×12 grid of effect sizes (z-scores under the null).  Cells that survive
BH correction at q=0.05 are outlined in black.

Saved to `results/figures/phase3_diagram_distances_per_domain_effect_heatmap.{pdf,png}`.

## save_fig helper

Verbatim from phase3_comparison.ipynb cell 9 — same export convention (PDF + PNG, results/figures/).

In [ ]:
def save_fig(fig, stem: str) -> None:
    """Export a figure as both .pdf (vector) and .png (300 dpi) to FIGURES_DIR."""
    pdf_path = FIGURES_DIR / f'{stem}.pdf'
    png_path = FIGURES_DIR / f'{stem}.png'
    fig.savefig(pdf_path, bbox_inches='tight')
    fig.savefig(png_path, dpi=300, bbox_inches='tight')
    print(f'  Saved {pdf_path.name}  +  {png_path.name}')


print('save_fig helper defined.')

In [ ]:
# ── Build effect-size matrix (12 × 12) ────────────────────────────────────
effect_mat = np.full((N_LAYERS, N_HEADS), np.nan)
sig_mat    = np.zeros((N_LAYERS, N_HEADS), dtype=bool)

for _, row in per_head_df.iterrows():
    effect_mat[int(row['layer']), int(row['head'])] = row['effect_size']
    sig_mat[int(row['layer']), int(row['head'])]    = bool(row['passes_bh'])

# ── Figure ─────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 8))

vmax = np.nanmax(np.abs(effect_mat))
im = ax.imshow(
    effect_mat,
    cmap='RdBu_r',
    vmin=-vmax,
    vmax=vmax,
    aspect='auto',
    interpolation='nearest',
)

# Outline BH-significant cells in black
for layer in range(N_LAYERS):
    for head in range(N_HEADS):
        if sig_mat[layer, head]:
            rect = mpatches.Rectangle(
                (head - 0.5, layer - 0.5), 1, 1,
                linewidth=1.5, edgecolor='black', facecolor='none',
            )
            ax.add_patch(rect)

cbar = fig.colorbar(im, ax=ax, shrink=0.85)
cbar.set_label('Effect size (z-score under null)', fontsize=11)

ax.set_xlabel('Attention head', fontsize=11)
ax.set_ylabel('Layer', fontsize=11)
ax.set_xticks(range(N_HEADS))
ax.set_yticks(range(N_LAYERS))
ax.set_xticklabels([str(h) for h in range(N_HEADS)])
ax.set_yticklabels([str(l) for l in range(N_LAYERS)])

n_sig = sig_mat.sum()
ax.set_title(
    f'Per-(layer, head) permutation test effect sizes\n'
    f'W₂ distance, H₁ — color domain (EN/RU/ES)\n'
    f'{n_sig}/144 cells significant at BH q={FDR_Q} (black outlines)',
    fontsize=12,
)

plt.tight_layout()
save_fig(fig, 'phase3_diagram_distances_per_domain_effect_heatmap')
plt.close(fig)

## Save summary CSV

Writes one row per `(layer, head)` plus an aggregate `"head-averaged"` row
to `results/phase3_diagram_distances/per_domain_perm.csv`.

In [ ]:
# ── Assemble summary CSV ───────────────────────────────────────────────────
csv_rows = per_head_df[['layer', 'head', 'observed', 'p_value', 'effect_size', 'passes_bh']].copy()

# Add an aggregate row for the head-averaged test
agg_row = pd.DataFrame([{
    'layer':       'agg',
    'head':        'agg',
    'observed':    agg_result['observed'],
    'p_value':     agg_result['p_value'],
    'effect_size': agg_result['effect_size'],
    'passes_bh':   agg_result['p_value'] < FDR_Q,  # single test, no BH needed
}])

df_out = pd.concat([csv_rows, agg_row], ignore_index=True)

out_path = DD_DIR / 'per_domain_perm.csv'
df_out.to_csv(out_path, index=False)
print(f'Written: {out_path}  ({len(df_out)} rows)')
print(df_out.tail(5).to_string(index=False))

## Per-term test (blr.4)

*Placeholder — subtask blr.4 will append cells here.*

Per-term permutation test: for each color term in the cross-linguistic
translation table, test whether the W_2 distances between the term's samples
across languages differ from within-language term distances.

Includes the Russian-blues design: синий vs. голубой split (Paramei 2005;
Winawer et al. 2007) — see `bd show ph-project-blr.4`.

## Per-head aggregation + 12×12 effect heatmap (blr.5)

*Placeholder — subtask blr.5 will append cells here.*

Rank (layer, head) cells by effect size; produce the final
`phase3_diagram_distances_per_head_effect_heatmap.{pdf,png}`.
Cross-reference Clark et al. 2019 head specialization qualitatively.

## Robustness: bottleneck + H_0 (blr.6)

*Placeholder — subtask blr.6 will append cells here.*

Re-run per-domain, per-term, and per-head tests using bottleneck distance
and H_0 homology dimension.  Reuses cached tensors from blr.2 — no new
computation needed.

## Discussion (blr.7)

*Placeholder — subtask blr.7 will add the final discussion cell here.*

Will tie findings to the hypothesis, summarize per-domain, per-term, and
Russian-blues results, note robustness, acknowledge limitations
(subsampling, фиолетовый under-target, color-only scope, single model),
and suggest follow-up (emotion + kinship, XLM-R, other Berlin-Kay languages).